# Clase 1 — Introducción a Pandas para Visualización de Datos

En esta clase en vivo vamos a repasar dos ideas fundamentales antes de escribir un solo gráfico:

1. **¿Por qué visualizar datos?** — la teoría y la intuición.
2. **Domar datos con Pandas** — el trabajo previo (preprocesamiento) que hace posible cualquier visualización útil.

Vamos a trabajar con un dataset real: la inmigración a Canadá entre 1980 y 2013 (fuente: Naciones Unidas).

## Parte 1 — ¿Por qué visualizar datos?

Imagina que te dan una tabla con miles de filas y te piden: "dime qué está pasando aquí". Mirando los números fila por fila tardarías horas y aun así se te escaparían los patrones. Tu cerebro está diseñado para reconocer **formas, colores y tendencias** de un vistazo, no para procesar números sueltos.

**Idea clave:** una tabla te da los datos, un gráfico te da la historia que esos datos cuentan.

Antes de analizar o modelar nada, visualizar te ayuda a:
- Detectar errores (valores raros, huecos, duplicados).
- Entender la forma general de los datos (¿sube?, ¿baja?, ¿hay grupos?).
- Decidir qué preguntas tiene sentido hacerle a los datos.

### Distintas preguntas piden distintos gráficos

No existe "el gráfico correcto" en abstracto, existe el gráfico correcto para la pregunta que haces:

| Pregunta | Tipo de gráfico |
|---|---|
| ¿Cómo cambió algo en el tiempo? | Línea |
| ¿Cómo se comparan categorías? | Barras |
| ¿Cómo se reparte un total en partes? | Pastel / treemap |
| ¿Qué relación hay entre dos variables? | Scatter (dispersión) |
| ¿Dónde pasa algo geográficamente? | Mapa |

Elegir mal el gráfico no es solo estética: puede hacer que la audiencia entienda algo que los datos no dicen.

### La caja de herramientas del curso

| Librería | ¿Para qué sirve? |
|---|---|
| **Matplotlib** | Base de graficado en Python. Mucho control, hay que construir cada detalle. |
| **Pandas** | La misma librería con la que cargas y limpias datos también grafica directo: `df.plot(kind='line')`. |
| **Seaborn** | Gráficos estadísticos con mejor estética por defecto. |
| **Plotly** | Visualizaciones interactivas (zoom, tooltips) — base de los dashboards con Dash. |
| **Folium** | Mapas interactivos (marcadores, mapas coropléticos). |
| **PyWaffle** | Gráficos de "waffle" — alternativa visual al pastel para mostrar proporciones. |

No necesitas dominarlas todas hoy: solo saber que existen y para qué sirve cada una.

### Puntos clave de la Parte 1

- Una tabla y una visualización no son lo mismo: la tabla contiene los datos, la visualización te ayuda a entenderlos.
- La pregunta que quieres responder determina el tipo de gráfico (tendencia → línea, comparación → barras, proporción → pastel, relación → scatter, ubicación → mapa).
- Con Pandas, un primer gráfico puede ser una sola línea: `df.plot(kind='line')`.

Ahora sí, antes de graficar nada, vamos a **domar los datos con Pandas**.

## Parte 2 — Domar tus datos con Pandas antes de graficar

Puedes conocer todos los trucos de Matplotlib, pero si le pasas datos sucios el gráfico resultante es, en el mejor de los casos, inútil, y en el peor, engañoso.

**Analogía:** tus datos son como ingredientes recién traídos del mercado — vienen con tierra pegada, hojas que no vas a usar, sin cortar. Pandas es tu tabla de cocina: ahí lavas (quitas columnas inútiles), cortas (seleccionas lo que interesa), organizas en recipientes (indexas) y separas por tipo (filtras y agrupas). Solo cuando los ingredientes están listos tiene sentido "cocinar" el gráfico.

Vamos a hacer este proceso en vivo con el dataset de inmigración a Canadá (1980-2013).

### Instalar dependencias

`pandas` necesita el paquete `openpyxl` para leer archivos Excel (`.xlsx`). En Google Colab lo instalamos con `pip`:

In [ ]:
!pip install openpyxl

### Cargar los datos

Importamos `pandas` (manejo de tablas) y `numpy` (cómputo científico), y cargamos el archivo Excel `Canada.xlsx` directamente desde su URL. Saltamos las primeras 20 filas (metadatos que no sirven) y las últimas 2 filas (totales resumen).

In [ ]:
import numpy as np   # útil para cómputo científico
import pandas as pd  # la librería principal para manejar datos en tablas

df_can = pd.read_excel(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.xlsx',
    sheet_name='Canada by Citizenship',
    skiprows=range(20),   # saltamos las primeras 20 filas (no son datos)
    skipfooter=2)         # ignoramos las últimas 2 filas (totales resumen)

print('¡Datos cargados en un DataFrame de pandas!')

### Explorar antes de tocar nada

Nunca empieces a modificar un dataset sin mirarlo primero. `df_can.shape` debería ser la primera línea de código que escribas con cualquier dataset nuevo.

In [ ]:
# Ver las primeras 5 filas
df_can.head()

In [ ]:
# Ver las últimas 5 filas
df_can.tail()

In [ ]:
# Resumen corto: cuántas columnas, tipos de datos, memoria usada
df_can.info(verbose=False)

In [ ]:
# Dimensiones: (número de filas, número de columnas)
df_can.shape

In [ ]:
# Estadísticas descriptivas de las columnas numéricas
df_can.describe()

### Limpiar: quitar y renombrar columnas

El dataset trae columnas (`AREA`, `REG`, `DEV`, `Type`, `Coverage`) que son metadatos de la fuente y no aportan nada a nuestro análisis. Las eliminamos con `.drop()`, y luego renombramos las columnas con nombres poco claros usando `.rename()`.

In [ ]:
# axis=0 son filas (por defecto), axis=1 son columnas
df_can.drop(['AREA', 'REG', 'DEV', 'Type', 'Coverage'], axis=1, inplace=True)
df_can.head(2)

In [ ]:
df_can.rename(columns={'OdName': 'Country', 'AreaName': 'Continent', 'RegName': 'Region'}, inplace=True)
df_can.columns

### Crear una columna nueva

Sumamos todos los años (1980 a 2013) fila por fila para obtener el total de inmigrantes por país.

In [ ]:
df_can['Total'] = df_can.sum(axis=1, numeric_only=True)
df_can['Total']

In [ ]:
# Revisamos si hay valores nulos
df_can.isnull().sum()

### Indexar: `.set_index()`, `.loc` y `.iloc`

Por defecto el índice es un rango numérico (0, 1, 2, ...), lo cual hace tedioso buscar un país específico. Convertimos la columna `Country` en el índice.

In [ ]:
df_can.set_index('Country', inplace=True)

# opcional: quitar el nombre "Country" del índice, es solo estético
df_can.index.name = None

df_can.head(3)

Con el país como índice tenemos dos formas de seleccionar filas:

- **`.loc[etiqueta]`** selecciona por el nombre/etiqueta del índice o columna.
- **`.iloc[posición]`** selecciona por posición numérica.

`.loc` es más legible; `.iloc` es útil cuando programas de forma genérica.

In [ ]:
# 1. Toda la fila de Japón (todas las columnas)
df_can.loc['Japan']

In [ ]:
# mismo resultado por posición (Japón está en la posición 87)
df_can.iloc[87]

In [ ]:
# 2. Solo el año 2013 para Japón
df_can.loc['Japan', 2013]

In [ ]:
# equivalente por posición: 2013 es la columna en la posición 36
df_can.iloc[87, 36]

In [ ]:
# 3. Un rango de años para Japón
df_can.loc['Japan', [1980, 1981, 1982, 1983, 1984, 1985]]

Los años en las columnas son de tipo `int`. Para evitar ambigüedades (¿"2013" es un año o una posición?), conviene convertir los nombres de columna a texto.

In [ ]:
df_can.columns = list(map(str, df_can.columns))

# variable útil que usaremos en los próximos capítulos para graficar
years = list(map(str, range(1980, 2014)))
years[:5]

### Filtrar filas con condiciones booleanas

Para quedarte solo con las filas que cumplen una condición, pasas una expresión booleana entre corchetes. Puedes combinar condiciones con `&` (y) y `|` (o) — nunca con `and`/`or` de Python puro — y siempre entre paréntesis.

In [ ]:
# 1. Creamos la condición (una serie de True/False, una por fila)
condition = df_can['Continent'] == 'Asia'

# 2. Se la pasamos al DataFrame para filtrar
df_can[condition].head()

In [ ]:
# Países de Asia que además pertenecen a la región "Southern Asia"
df_can[(df_can['Continent'] == 'Asia') & (df_can['Region'] == 'Southern Asia')]

### Agrupar con `groupby`

`groupby` responde preguntas del tipo "¿cuál es el total por categoría?".

In [ ]:
df_country = df_can.groupby(['Continent'])['Total'].sum().reset_index()
df_country

### Ordenar con `sort_values`

Para encontrar los países que más contribuyeron al total de inmigración, ordenamos por la columna `Total`.

In [ ]:
df_can.sort_values(by='Total', ascending=False, axis=0, inplace=True)
top_5 = df_can.head(5)
top_5

### Puntos clave de la Parte 2

- Ningún gráfico arregla datos mal preparados: limpia y explora primero, grafica después.
- `.head()`, `.tail()`, `.info()`, `.shape`, `.describe()` son tu primer contacto con cualquier dataset nuevo.
- `.drop()` elimina columnas/filas; `.rename()` les da nombres con sentido.
- `.set_index()` facilita buscar por nombre en vez de por posición numérica.
- `.loc` selecciona por etiqueta; `.iloc` por posición. No son intercambiables.
- Filtrar es pasar una condición booleana entre corchetes: `df[condicion]`, combinando con `&`/`|` entre paréntesis.
- `groupby()` agrupa y agrega; `sort_values()` ordena el resultado.

En la práctica de hoy vas a aplicar todo esto para responder preguntas concretas sobre este mismo dataset.